In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
spark = SparkSession.builder \
    .appName("Data Mining Spark Data Lake") \
    .getOrCreate()

In [ ]:
import os
import shutil

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

In [ ]:
spark = SparkSession.builder \
    .appName("Data Mining Spark Data Lake") \
    .getOrCreate()

print("SparkSession created successfully!")

SparkSession created successfully!


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'sales.csv', 'data_lake', 'silver', 'sample_data']


In [ ]:
bronze_path = "data_lake/bronze/sales.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [ ]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [ ]:
initial_count = df.count()

print("Total records:", initial_count)

Total records: 1000


In [ ]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 10


In [ ]:
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [ ]:
df.select("category") \
    .distinct() \
    .orderBy("category") \
    .show(truncate=False)

+---------------+
|category       |
+---------------+
|Beauty         |
|Books          |
|CLOTHING       |
|Clothing       |
|ELECTRONICS    |
|Electronics    |
|Furniture      |
|Grocery        |
|Home Appliances|
|SPORTS         |
|Sports         |
|beauty         |
|sports         |
+---------------+



In [ ]:
df.select("city") \
    .distinct() \
    .orderBy("city") \
    .show(truncate=False)

+-----------+
|city       |
+-----------+
|NULL       |
| Hyderabad |
| Mumbai    |
|Bengaluru  |
|Chennai    |
|Coimbatore |
|DELHI      |
|Delhi      |
|HYDERABAD  |
|Hyderabad  |
|JAIPUR     |
|Jaipur     |
|Kolkata    |
|Lucknow    |
|Mumbai     |
|Pune       |
|Vijayawada |
|coimbatore |
|delhi      |
+-----------+



In [ ]:
invalid_quantity = df.filter(
    col("quantity") <= 0
)

print("Invalid quantity records:", invalid_quantity.count())

invalid_quantity.show(truncate=False)

Invalid quantity records: 12
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100114  |C0252      |Keerthi Singh|Ceiling Fan           |Home Appliances|0       |2920      |15              |Net Banking     |Chennai   |Tamil Nadu    |2025-02-15|Delivered   |
|100143  |C0184      |Aman Joshi   |Football              |Sports         |0       |1030      |15              |Debit Card      |Coimbatore|Tamil Nadu    |2025-11-26|Delivered   |
|100198  |C0014      |Arjun Pillai |Hoodie                |Clothing    

In [ ]:
invalid_price = df.filter(
    col("unit_price") <= 0
)

print("Invalid unit price records:", invalid_price.count())

invalid_price.show(truncate=False)

Invalid unit price records: 8
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery    |7       |0         |40              |Credit Card   |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100170  |C0158      |Deepak Pandey   |Competitive Exam Guide|Books      |6       |0         |0               |Net Banking   |Pune      |Maharashtra   |2025-08-26|Shipped     |
|100235  |C0349      |Tanvi Reddy     |Power Bank            |Electronics|7       |-5

In [ ]:
invalid_discount = df.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
)

print("Invalid discount records:", invalid_discount.count())

invalid_discount.show(truncate=False)

Invalid discount records: 8
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|customer_name  |product         |category       |quantity|unit_price|discount_percent|payment_method  |city      |state     |order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|100034  |C0192      |Divya Sharma   |Yoga Mat        |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|Delivered   |
|100140  |C0075      |Swathi Tripathi|Cooking Oil 5L  |Grocery        |4       |790       |-5              |Credit Card     |Hyderabad |Telangana |2025-10-01|Shipped     |
|100217  |C0214      |Neha Tripathi  |Football        |Sports         |1       |650       |-5              |UPI 

In [ ]:
date_check = df.withColumn(
    "parsed_date",
    coalesce(
        to_date(expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")),
        to_date(expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")),
        to_date(expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")),
        to_date(expr("try_to_timestamp(order_date, 'MM-dd-yyyy')"))
    )
)

invalid_dates = date_check.filter(
    col("parsed_date").isNull()
)

print("Invalid date records:", invalid_dates.count())

invalid_dates.select(
    "order_id",
    "order_date"
).show(truncate=False)

Invalid date records: 4
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
+--------+----------+



In [ ]:
silver_df = df.dropDuplicates()

print("Rows before preprocessing:", df.count())
print("Rows after removing duplicates:", silver_df.count())

Rows before preprocessing: 1000
Rows after removing duplicates: 990


In [ ]:
text_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_date",
    "order_status"
]

for c in text_columns:
    silver_df = silver_df.withColumn(
        c,
        trim(col(c))
    )

In [ ]:
silver_df = silver_df.withColumn(
    "category",
    initcap(lower(col("category")))
)

In [ ]:
silver_df = silver_df.withColumn(
    "city",
    initcap(lower(col("city")))
)

In [ ]:
silver_df = silver_df.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

In [ ]:
valid_quantity = silver_df.filter(
    col("quantity") > 0
)

quantity_median = valid_quantity.approxQuantile(
    "quantity",
    [0.5],
    0.0
)[0]

print("Median valid quantity:", quantity_median)

Median valid quantity: 5.0


In [ ]:
silver_df = silver_df.withColumn(
    "quantity",
    when(
        col("quantity") <= 0,
        lit(quantity_median)
    ).otherwise(col("quantity"))
)

In [ ]:
valid_price = silver_df.filter(
    col("unit_price") > 0
)

price_median = valid_price.approxQuantile(
    "unit_price",
    [0.5],
    0.0
)[0]

print("Median valid unit price:", price_median)

Median valid unit price: 1850.0


In [ ]:
silver_df = silver_df.withColumn(
    "unit_price",
    when(
        col("unit_price") <= 0,
        lit(price_median)
    ).otherwise(col("unit_price"))
)

In [ ]:
valid_discount = silver_df.filter(
    (col("discount_percent") >= 0) &
    (col("discount_percent") <= 100)
)

discount_median = valid_discount.approxQuantile(
    "discount_percent",
    [0.5],
    0.0
)[0]

print("Median valid discount:", discount_median)

Median valid discount: 20.0


In [ ]:
silver_df = silver_df.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") < 0) |
        (col("discount_percent") > 100),
        lit(discount_median)
    ).otherwise(col("discount_percent"))
)

In [ ]:
silver_df = silver_df.withColumn(
    "order_date",
    coalesce(
        to_date(expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")),
        to_date(expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")),
        to_date(expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")),
        to_date(expr("try_to_timestamp(order_date, 'MM-dd-yyyy')"))
    )
)

In [ ]:
silver_df = silver_df.withColumn(
    "order_date",
    date_format(col("order_date"), "yyyy-MM-dd")
)

In [ ]:
silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [ ]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
final_count = silver_df.count()

print("Rows before preprocessing:", initial_count)
print("Rows after preprocessing:", final_count)
print("Rows removed:", initial_count - final_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [ ]:
remaining_duplicates = (
    silver_df.count() -
    silver_df.dropDuplicates().count()
)

print("Remaining duplicate rows:", remaining_duplicates)

Remaining duplicate rows: 0


In [ ]:
print(
    "Invalid quantities:",
    silver_df.filter(
        col("quantity") <= 0
    ).count()
)

print(
    "Invalid unit prices:",
    silver_df.filter(
        col("unit_price") <= 0
    ).count()
)

print(
    "Invalid discounts:",
    silver_df.filter(
        (col("discount_percent") < 0) |
        (col("discount_percent") > 100)
    ).count()
)

Invalid quantities: 0
Invalid unit prices: 0
Invalid discounts: 0


In [ ]:
silver_path = "data_lake/silver/preprocessed_sales"

silver_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Silver data saved successfully!")

Silver data saved successfully!


In [ ]:
silver_read = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(silver_path)

print("Silver data loaded successfully!")

Silver data loaded successfully!


In [ ]:
silver_read.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [ ]:
print("Silver row count:", silver_read.count())

Silver row count: 990


In [ ]:
import glob
import shutil
import os

os.makedirs("silver", exist_ok=True)

part_file = glob.glob(
    "data_lake/silver/preprocessed_sales/part-*.csv"
)[0]

shutil.copy(
    part_file,
    "silver/sales_silver.csv"
)

print("Created silver/sales_silver.csv")

Created silver/sales_silver.csv


In [ ]:
print("FINAL VERIFICATION")
print("==================")
print("Bronze records:", initial_count)
print("Silver records:", silver_read.count())
print("Duplicate rows remaining:", remaining_duplicates)
print(
    "Silver file exists:",
    os.path.exists("silver/sales_silver.csv")
)

FINAL VERIFICATION
Bronze records: 1000
Silver records: 990
Duplicate rows remaining: 0
Silver file exists: True
